# Customer Segmentation with K-Means and PCA (q2_unsupervised)

## 1. Data Preparation

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

df = pd.read_csv('q2_customers.csv')
print("Shape:", df.shape)
df.head()


**Why scaling is essential:** K-Means uses Euclidean distance. Features with larger numeric ranges would dominate the distance calculation if data were unscaled. StandardScaler gives each feature comparable influence by centering to mean 0 and scaling to unit variance.

In [ ]:

features = df.columns.tolist()
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## 2. Choosing K — Elbow Method

In [ ]:

wcss = []
K_range = range(1,11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(list(K_range), wcss, marker='o')
plt.xticks(list(K_range))
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.show()


**Interpretation:** Choose the K where the decrease in WCSS begins to slow sharply (the elbow). That point balances compact clusters with model simplicity. Inspect the plot and select the most visible bend.

In [ ]:

# Set chosen K after viewing elbow plot
chosen_k = 4
print("Chosen K =", chosen_k)


## 3. K-Means Clustering

In [ ]:

kmeans = KMeans(n_clusters=chosen_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df['cluster'] = clusters

centroids_scaled = pd.DataFrame(kmeans.cluster_centers_, columns=features)
centroids_original = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)

print("Cluster Counts:")
print(df['cluster'].value_counts().sort_index())

print("\nCentroids (Original Scale):")
centroids_original


**Business Interpretation:** Review centroid values. For example, clusters may represent high spend frequent shoppers, infrequent bargain shoppers, loyal medium spenders, or recently inactive customers. Use age, spend, visits, and basket size to label each segment.

## 4. Dimensionality Reduction with PCA

In [ ]:

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Explained Variance Ratio:")
print(pca.explained_variance_ratio_)

loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=features
)
print("\nFeature Loadings:")
loadings


**Interpretation:** Features with large positive/negative loadings contribute most to each principal component. PC1 often captures overall spending/activity intensity, while PC2 may separate demographics or recency behavior depending on loadings.

## 5. Cluster Visualisation

In [ ]:

plot_df = pd.DataFrame({
    'PC1': X_pca[:,0],
    'PC2': X_pca[:,1],
    'cluster': clusters
})

plt.figure(figsize=(8,6))
sns.scatterplot(data=plot_df, x='PC1', y='PC2', hue='cluster', palette='tab10', s=60)
plt.title('Customer Segments Visualised with PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.show()
